# Stage 5 — Holdout Test  ⚠️ RUN ONCE

**This notebook executes exactly once in the project lifecycle.**

Re-running invalidates the result. The Holdout set is the sole authoritative measurement of out-of-sample performance.

**Acceptance criteria (all must hold):**

| Metric | Threshold |
|--------|-----------|
| Profit Factor | > 1.3 |
| Sharpe Ratio | > 1.0 |
| Max Drawdown | < 30% |
| Deflated Sharpe (N=1500) | > 0 |

Failure → strategy retired, postmortem in `results/POSTMORTEM.md`.

In [ ]:
from __future__ import annotations

import json
import pickle
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from quant.backtest.costs import CostMode, get_cost_model
from quant.backtest.metrics import compute_metrics
from quant.strategies.mean_reversion import MeanReversionConfig, MeanReversionStrategy
from quant.validation.deflated_sharpe import deflated_sharpe_ratio

DATA_PROC = Path('../data/processed')
RESULTS = Path('../results')

PF_MIN = 1.3
SHARPE_MIN = 1.0
MAX_DD_MAX = 0.30
N_TRIALS_LOCKED = 1500

In [ ]:
with open(DATA_PROC / 'split_BTCUSDT_1h.pkl', 'rb') as f:
    split = pickle.load(f)
chosen = json.loads((DATA_PROC / 'chosen_config.json').read_text())
params = chosen['params']

base = MeanReversionConfig(name='mean_reversion', timeframe='1h')
config = MeanReversionConfig(**{**asdict(base), **params})

print(f'Holdout bars: {len(split.holdout_data):,}  [{split.holdout_range[0]}  →  {split.holdout_range[1]}]')
print('Config:')
print(json.dumps(params, indent=2))

In [ ]:
strat = MeanReversionStrategy(config)
trades = strat.simulate(split.holdout_data, cost_model=get_cost_model(CostMode.HORROR))
metrics = compute_metrics(trades)

for k, v in metrics.items():
    print(f'{k:20s}: {v}')

In [ ]:
# Deflated Sharpe — correcting for the 1500 grid search trials.
# sharpe_std is the std dev of Sharpe across the original 1500 trials.
is_log = pd.read_csv(RESULTS / 'tables' / 'is_search_log.csv')
sharpe_std = float(is_log['sharpe'].std())

# n_returns = number of daily returns in the holdout period
daily = trades.set_index('entry_time')['net_pct'].resample('1D').sum().fillna(0.0)
n_returns = len(daily)
skew = float(daily.skew())
kurt = float(daily.kurtosis() + 3.0)  # scipy convention = excess kurtosis + 3

dsr = deflated_sharpe_ratio(
    observed_sharpe=metrics['sharpe'],
    sharpe_std=sharpe_std,
    n_trials=N_TRIALS_LOCKED,
    n_returns=n_returns,
    skewness=skew,
    kurtosis=kurt,
)
print(f'Sharpe (observed):     {metrics["sharpe"]:.4f}')
print(f'Sharpe std (1500 IS):  {sharpe_std:.4f}')
print(f'Daily returns:         {n_returns}')
print(f'Skewness:              {skew:.3f}')
print(f'Kurtosis:              {kurt:.3f}')
print(f'Deflated Sharpe Ratio: {dsr:.4f}  (must be > 0 to pass)')

In [ ]:
gates = {
    'pf':          (metrics['pf']     > PF_MIN,     f"{metrics['pf']:.3f}  > {PF_MIN}"),
    'sharpe':      (metrics['sharpe'] > SHARPE_MIN, f"{metrics['sharpe']:.3f}  > {SHARPE_MIN}"),
    'max_dd':      (metrics['max_dd'] < MAX_DD_MAX, f"{metrics['max_dd']:.3f}  < {MAX_DD_MAX}"),
    'dsr':         (dsr > 0,                        f"{dsr:.3f}  > 0"),
}
print('Acceptance gates:')
for name, (passed, expr) in gates.items():
    mark = '✓' if passed else '✗'
    print(f'  {mark}  {name:8s}  {expr}')

all_pass = all(p for p, _ in gates.values())
print(f'\nStage 5 verdict: {"PASS" if all_pass else "FAIL — strategy retired"}')

# Persist final holdout metrics
holdout_record = {
    **metrics,
    'dsr': dsr,
    'gates_passed': all_pass,
    'params': params,
}
(RESULTS / 'tables').mkdir(parents=True, exist_ok=True)
(RESULTS / 'tables' / 'holdout_result.json').write_text(json.dumps(holdout_record, indent=2, default=str))

In [ ]:
equity = (1.0 + daily).cumprod()
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(equity.index, equity, color='#3b82f6', lw=1.2)
ax.axhline(1.0, color='gray', lw=0.5)
ax.set_title(f'Holdout equity curve — Sharpe {metrics["sharpe"]:.2f} / PF {metrics["pf"]:.2f} / MaxDD {metrics["max_dd"]:.1%}')
ax.set_ylabel('Equity (start = 1.0)')
ax.grid(alpha=0.3)

out = RESULTS / 'figures' / '06_holdout_equity.png'
fig.savefig(out, dpi=120, bbox_inches='tight')
print(f'Wrote: {out}')
plt.show()